In [ ]:
!git clone https://github.com/binsec/rosa.git

Cloning into 'rosa'...
remote: Enumerating objects: 2832, done.
remote: Counting objects: 100% (2832/2832), done.
remote: Compressing objects: 100% (953/953), done.
remote: Total 2832 (delta 1795), reused 2746 (delta 1710), pack-reused 0 (from 0)
Receiving objects: 100% (2832/2832), 1.20 MiB | 6.14 MiB/s, done.
Resolving deltas: 100% (1795/1795), done.


In [ ]:
%cd rosa

/content/rosa


In [ ]:
!ls -la

total 192
drwxr-xr-x 8 root root  4096 Jul 31 19:14 .
drwxr-xr-x 1 root root  4096 Jul 31 19:14 ..
-rw-r--r-- 1 root root   181 Jul 31 19:14 AUTHORS
-rwxr-xr-x 1 root root   932 Jul 31 19:14 build.sh
-rw-r--r-- 1 root root 57099 Jul 31 19:14 Cargo.lock
-rw-r--r-- 1 root root   845 Jul 31 19:14 Cargo.toml
-rw-r--r-- 1 root root   174 Jul 31 19:14 CHANGELOG.md
-rw-r--r-- 1 root root  3129 Jul 31 19:14 CITATION.cff
-rw-r--r-- 1 root root  2199 Jul 31 19:14 CONTRIBUTING.md
drwxr-xr-x 3 root root  4096 Jul 31 19:14 doc
-rw-r--r-- 1 root root  2397 Jul 31 19:14 Dockerfile
-rw-r--r-- 1 root root   109 Jul 31 19:14 .dockerignore
drwxr-xr-x 3 root root  4096 Jul 31 19:14 examples
drwxr-xr-x 3 root root  4096 Jul 31 19:14 fuzzers
drwxr-xr-x 8 root root  4096 Jul 31 19:14 .git
drwxr-xr-x 3 root root  4096 Jul 31 19:14 .github
-rw-r--r-- 1 root root    59 Jul 31 19:14 .gitignore
-rw-r--r-- 1 root root   249 Jul 31 19:14 .gitmodules
-rw-r--r-- 1 root root     5 Jul 31 19:14 IMAGE
-rw-r--r-- 1 root 

In [ ]:
!cat README.md

# ROSA: Finding Backdoors with Fuzzing

[![Paper DOI badge](https://img.shields.io/badge/Paper%20DOI-10.1109%2FICSE55347.2025.00183-blue?style=flat)](https://doi.org/10.1109/ICSE55347.2025.00183)
[![Zenodo DOI badge](https://img.shields.io/badge/Zenodo%20DOI-10.5281%2Fzenodo.14724250-blue?style=flat)](https://doi.org/10.5281/zenodo.14724250)
[![SWH](https://archive.softwareheritage.org/badge/origin/https://github.com/binsec/rosa/)](https://archive.softwareheritage.org/browse/origin/?origin_url=https://github.com/binsec/rosa)

## About

ROSA[^1] is a fuzzing-based toolchain for backdoor detection in binary programs. It uses a
state-of-the-art fuzzer ([AFL++](https://github.com/AFLplusplus/AFLplusplus)) coupled with a novel
[metamorphic oracle](https://en.wikipedia.org/wiki/Metamorphic_testing) to detect many different
types of backdoors in different types of binary programs.

A presentation of ROSA (including a live demo) was given at FOSDEM'26:
<https://mirrors.dotsrc.org/fosdem/2026/u

In [ ]:
!cat build.sh

#!/usr/bin/env bash

## Build Docker image for the ROSA toolchain.
## The name of the Docker image is specified by the IMAGE file.
## The version of the Docker image is specified by the VERSION file.


set -e

# The command `git submodule status` displays the list of registered submodules in the current
# repo. If a submodule is not cloned/uninitialized, its corresponding line in the command's output
# is prefixed with a '-'. So, by looking at the first byte, we can tell if any submodule is not
# cloned and stop the build.
status_list=$(git submodule status --recursive | cut -b 1)
for status in $status_list
do
    if [ "$status" == "-" ]
    then
        echo "At least one submodule is uninitialized; stopping build." 1>&2
        echo "Run \`git submodule update --init --recursive\` at the root of the repo." 1>&2
        exit 1
    fi
done

docker build -t $(cat IMAGE):$(cat VERSION) . --label "version=$(cat VERSION)"


In [ ]:
!cat run.sh

#!/usr/bin/env bash

## Run a Docker container with the ROSA toolchain image.
## The name of the Docker image is specified by the IMAGE file.
## The version of the Docker image is specified by the VERSION file.


set -e

docker run -ti --rm -p 4000:4000 $(cat IMAGE):$(cat VERSION)
